CELL 1: Install Dependencies


In [1]:
!pip install faiss-cpu openai nltk numpy requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 71.3 MB/s eta 0:00:00


CELL 2: Imports

In [2]:
import numpy as np
import faiss
import nltk
from nltk.tokenize import sent_tokenize
from openai import OpenAI
from google.colab import userdata
nltk.download("punkt")


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

CELL 3: Load API Credentials from Colab

In [4]:
API_KEY = userdata.get("api_key")
BASE_URL = userdata.get("BASE_URL")

if not API_KEY or not BASE_URL:
    raise RuntimeError("❌ API_KEY / BASE_URL missing in Colab Secrets")

client = OpenAI(api_key=API_KEY, base_url=BASE_URL)
print("✅ API loaded successfully")


✅ API loaded successfully


CELL 4: Load External Document

In [6]:
from google.colab import files

# Upload document
uploaded = files.upload()

# Get uploaded filename automatically
filename = list(uploaded.keys())[0]

# Read the document
with open(filename, "r", encoding="utf-8") as f:
    raw_text = f.read()

print("Uploaded file:", filename)
print("📄 Document length:", len(raw_text))
print("\nPreview:")
print(raw_text[:500])


Saving AI_Budget_Shopping_Assistant_RAG_Knowledge_Base.txt to AI_Budget_Shopping_Assistant_RAG_Knowledge_Base.txt
Uploaded file: AI_Budget_Shopping_Assistant_RAG_Knowledge_Base.txt
📄 Document length: 6064

Preview:
AI BUDGET SHOPPING ASSISTANT — RAG KNOWLEDGE BASE
Synthetic demonstration data for RAG ingestion. Refresh prices/availability from a live product API in production.

=== PRODUCT CATALOG ===
PRODUCT_ID: BP101
NAME: UrbanPack College Backpack
CATEGORY: Backpack
PRICE_INR: 1499
RATING: 4.3
FEATURES: 25L; laptop compartment; water resistant; padded shoulder straps
USE_CASE: Suitable for college students and daily commuting.
---
PRODUCT_ID: BP102
NAME: TravelPro Laptop Backpack
CATEGORY: Backpack
PRI


CELL 5: Semantic Chunking

In [8]:
nltk.download("punkt_tab")
def semantic_chunking(text, max_sentences=5):
    sentences = sent_tokenize(text)
    chunks = []

    for i in range(0, len(sentences), max_sentences):
        chunk = " ".join(sentences[i:i + max_sentences])
        chunks.append(chunk)

    return chunks

documents = semantic_chunking(raw_text)
print("🧩 Total semantic chunks:", len(documents))


🧩 Total semantic chunks: 9


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


CELL 6: API-Based Embeddings

In [9]:
def embed_texts(texts):
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=texts
    )
    return np.array([item.embedding for item in response.data], dtype="float32")

doc_embeddings = embed_texts(documents)
print("📐 Embedding shape:", doc_embeddings.shape)


📐 Embedding shape: (9, 1536)


In [10]:
dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(doc_embeddings)

print("📦 Indexed chunks:", index.ntotal)


📦 Indexed chunks: 9


Build Vector Database

In [11]:
def is_related_to_index(query, threshold=1.8):
    q_emb = embed_texts([query])
    distances, _ = index.search(q_emb, 1)
    return distances[0][0] < threshold


def retrieve(query, k=3):
    q_emb = embed_texts([query])
    _, idx = index.search(q_emb, k)
    return [documents[i] for i in idx[0]]


Is Query Related to Index?

In [12]:
def generate_answer(prompt):
    response = client.chat.completions.create(
        model="gpt-4.1-nano",
        messages=[
            {"role": "system", "content": "Answer strictly using the provided context only.,IF CONTENT IS NOT IN CONTEXT MEANS DONT PROVIDE"},
            {"role": "user", "content": prompt}
        ],
        temperature=0.3
    )
    return response.choices[0].message.content


In [13]:
def hallucinated(answer, chunks, threshold=0.6):
    texts = chunks + [answer]
    embeddings = embed_texts(texts)

    answer_emb = embeddings[-1]
    chunk_embs = embeddings[:-1]

    similarities = [
        np.dot(answer_emb, c) / (np.linalg.norm(answer_emb) * np.linalg.norm(c))
        for c in chunk_embs
    ]

    return max(similarities) < threshold


In [14]:
def preprocess_query(query):
    if len(query.split()) < 3 and not query.lower().startswith("what is"):
        return f"What is {query}?"
    return query


def rewrite_query(query):
    return f"Explain clearly with examples: {query}"


In [15]:
def web_search(query):
    response = client.chat.completions.create(
        model="gpt-4.1-nano",
        messages=[
            {
                "role": "system",
                "content": "You are a web-enabled assistant. Provide factual answers."
            },
            {
                "role": "user",
                "content": query
            }
        ],
        temperature=0.2
    )
    return response.choices[0].message.content


In [16]:
def adaptive_rag(query, max_loops=3):
    original_query = query

    for loop in range(1, max_loops + 1):
        print(f"\n🔁 LOOP {loop}: {query}")

        processed_query = preprocess_query(query)

        if not is_related_to_index(processed_query):
            print("❌ Not related to index → Web search fallback")
            return web_search(processed_query)

        docs = retrieve(processed_query)
        context = " ".join(docs)

        prompt = f"""
Use ONLY the context below.

Context:
{context}

Question:
{query}

Answer:
"""

        answer = generate_answer(prompt)

        if not hallucinated(answer, docs):
            print("✅ Answer verified")
            return answer

        print("⚠️ Hallucination detected → rewriting query")
        query = rewrite_query(original_query)

    return "⚠️ Unable to verify answer after multiple attempts."


In [22]:
result = adaptive_rag("what is my project about?")
print("\n🟢 FINAL ANSWER:\n", result)



🔁 LOOP 1: what is my project about 
✅ Answer verified

🟢 FINAL ANSWER:
 The project is about assisting with shopping for student-related products, including backpacks, headphones, smartwatches, and study accessories, by providing information on features, prices, and suitability for college use.
